# 00 · GPU and environment check

Run this first. It must end with **ALL CHECKS PASSED**. If PyTorch can't see the GPU, it stops with an error
that says what to fix. (Kernel: **PlantIQ compare (GPU)**.)

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("compare/ folder:", ROOT)
print("python:", sys.executable)
assert ".venv" in sys.executable, "Wrong kernel: select 'PlantIQ compare (GPU)' (compare/.venv) in the kernel picker."

In [ ]:
import time
import torch
from common import config, env

device = env.require_cuda()
props = torch.cuda.get_device_properties(0)
print(f"VRAM: {props.total_memory / 1024**3:.1f} GB | compute capability {props.major}.{props.minor}")

## GPU vs CPU speed (a 4096x4096 matrix multiply)

In [ ]:
def bench(dev, n=4096, reps=10):
    a = torch.randn(n, n, device=dev)
    b = torch.randn(n, n, device=dev)
    for _ in range(2):
        a @ b
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(reps):
        a @ b
    if dev.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t) / reps * 1000

gpu_ms = bench(device)
cpu_ms = bench(torch.device("cpu"), reps=2)
print(f"GPU {gpu_ms:.1f} ms | CPU {cpu_ms:.1f} ms | GPU is {cpu_ms / gpu_ms:.0f}x faster")
assert gpu_ms < cpu_ms, "GPU slower than CPU - something is wrong with the CUDA setup"

## A ResNet-50 training step on the GPU with mixed precision

In [ ]:
from torchvision import models

m = models.resnet50(weights=None).to(device)
opt = torch.optim.AdamW(m.parameters(), 1e-3)
scaler = torch.amp.GradScaler("cuda")
x = torch.randn(64, 3, 224, 224, device=device)
y = torch.randint(0, 5, (64,), device=device)
for i in range(11):
    if i == 1:
        torch.cuda.synchronize(); t = time.perf_counter()
    with torch.autocast("cuda", dtype=torch.float16):
        loss = torch.nn.functional.cross_entropy(m(x), y)
    opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
torch.cuda.synchronize()
ips = 64 * 10 / (time.perf_counter() - t)
print(f"ResNet-50 @224, batch 64: {ips:.0f} images/s training (expected ~350-400 on an RTX 3070 Laptop)")
assert ips > 150, ("GPU training is far too slow: plug in the charger, set Windows power mode to 'Best performance', "
                   "close other GPU programs, then run this notebook again.")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
del m, opt, x, y
torch.cuda.empty_cache()

## Packages and inputs

In [ ]:
import importlib

for pkg in ["torchvision", "sklearn", "scipy", "pandas", "matplotlib", "seaborn", "imagehash", "ultralytics",
            "sentence_transformers", "faiss", "rank_bm25", "fitz", "langchain_community", "langchain_text_splitters",
            "httpx", "dotenv"]:
    mod = importlib.import_module(pkg)
    print(f"{pkg:26s} {getattr(mod, '__version__', 'ok')}")

assert config.DATA_DIR.exists(), f"dataset not found at {config.DATA_DIR}"
counts = {p.name: sum(1 for _ in p.iterdir()) for p in sorted(config.DATA_DIR.iterdir()) if p.is_dir()}
print("dataset:", counts, "total", sum(counts.values()))
assert config.DEPLOYED_WEIGHTS.exists(), f"deployed weights not found at {config.DEPLOYED_WEIGHTS}"
print("deployed weights:", config.DEPLOYED_WEIGHTS.name, f"{config.DEPLOYED_WEIGHTS.stat().st_size / 1e6:.1f} MB")
print("smoke mode:", config.SMOKE, "| folds:", config.N_FOLDS)
print("\nALL CHECKS PASSED")